# Notebook 03: Análisis Exploratorio de Datos (EDA)
## Proyecto Integrador — Minería de Datos I

**Objetivo:** Responder a las preguntas de análisis formuladas en la inspección inicial mediante visualizaciones e interpretaciones. Cada gráfico debe vincularse a una pregunta concreta y aportar evidencia al análisis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# Cargar dataset procesado (limpio)
df = pd.read_csv('../data/processed/reporte_clinica_analisis.csv')
print(f'Dataset: {df.shape[0]} filas x {df.shape[1]} columnas')
print(f'Nulos: {df.isnull().sum().sum()}')
df.head()

## Preguntas de análisis

1. ¿Cómo se distribuyen las variables numéricas del dataset?
2. ¿Existen diferencias significativas en los costos entre fumadores y no fumadores?
3. ¿Cómo se relaciona la edad y el IMC con el costo del seguro?
4. ¿Hay diferencias regionales en los costos y en la proporción de fumadores?
5. ¿Qué combinación de factores explica mejor la variabilidad en los costos?

---
## 1. Análisis Univariado

### 1.1 Distribución de la variable objetivo: `charges`

**Pregunta:** ¿Cómo se distribuyen los costos del seguro médico?

**¿Por qué es relevante?** Identificar la forma de la distribución permite anticipar si hay segmentos de costo diferenciados y si la variable es adecuada para modelos lineales.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histograma + KDE
sns.histplot(df['charges'], kde=True, bins=40, color='steelblue', edgecolor='white', ax=axes[0])
axes[0].axvline(df['charges'].median(), color='red', linestyle='--', linewidth=2, label=f'Mediana: \${df["charges"].median():,.0f}')
axes[0].axvline(df['charges'].mean(), color='orange', linestyle='--', linewidth=2, label=f'Media: \${df["charges"].mean():,.0f}')
axes[0].set_title('Distribución de Costos de Seguro')
axes[0].set_xlabel('Costo (USD)')
axes[0].set_ylabel('Frecuencia')
axes[0].legend()

# Boxplot
sns.boxplot(x=df['charges'], color='lightblue', ax=axes[1])
axes[1].set_title('Boxplot de Costos')
axes[1].set_xlabel('Costo (USD)')

plt.tight_layout()
plt.show()

print(f'Estadísticos de charges:')
print(f'  Media:   \${df["charges"].mean():,.2f}')
print(f'  Mediana: \${df["charges"].median():,.2f}')
print(f'  Mín:     \${df["charges"].min():,.2f}')
print(f'  Máx:     \${df["charges"].max():,.2f}')
print(f'  Asimetría: {df["charges"].skew():.2f}')

**Interpretación:** La distribución de costos muestra una clara **asimetría positiva (skew = 1.51)**. La mayoría de los asegurados paga entre $2,000 y $15,000, pero existe un grupo reducido con costos muy elevados ($40,000–$65,000). La media ($13,256) es significativamente mayor que la mediana ($9,361), lo cual es un indicador de que un subconjunto pequeño de la población impulsa el costo promedio hacia arriba. Este comportamiento es típico en seguros de salud, donde condiciones preexistentes o factores de riesgo (como el tabaquismo) generan costos desproporcionadamente altos.

### 1.2 Distribución de `age` por grupo etario

**Pregunta:** ¿Qué grupo etario está más representado en el dataset?

**¿Por qué es relevante?** La edad es un factor de riesgo conocido en seguros médicos. Conocer la distribución etaria ayuda a contextualizar los resultados del análisis.

In [ ]:
# Crear grupos etarios
bins = [0, 25, 35, 45, 55, 100]
labels = ['18-25', '26-35', '36-45', '46-55', '56-64']
df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels, right=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histograma
sns.histplot(df['age'], bins=20, kde=True, color='teal', edgecolor='white', ax=axes[0])
axes[0].set_title('Distribución de Edad')
axes[0].set_xlabel('Edad')
axes[0].set_ylabel('Frecuencia')

# Barras por grupo etario
age_counts = df['age_group'].value_counts().sort_index()
colors = sns.color_palette('Blues_d', len(age_counts))
bars = axes[1].bar(age_counts.index.astype(str), age_counts.values, color=colors, edgecolor='black')
axes[1].set_title('Distribución por Grupo Etario')
axes[1].set_xlabel('Grupo Etario')
axes[1].set_ylabel('Frecuencia')
axes[1].tick_params(axis='x', rotation=45)

for bar, val in zip(bars, age_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, str(val), ha='center', fontsize=10)

plt.tight_layout()
plt.show()

print('Frecuencia por grupo etario:')
print(age_counts)

**Interpretación:** La distribución de edades es relativamente uniforme entre los 18 y 64 años, con una ligera mayor concentración en el rango de 18-25 años. No hay un sesgo etario marcado, lo cual sugiere que el dataset captura una muestra diversa en términos de edad. Esto es relevante porque cualquier conclusión sobre la relación edad-costo no estará distorsionada por una sobrerrepresentación de un grupo específico.

---
## 2. Análisis Bivariado

### 2.1 Costos según condición de fumador

**Pregunta:** ¿Existen diferencias significativas en los costos entre fumadores y no fumadores?

**¿Por qué es relevante?** El tabaquismo es un factor de riesgo determinante en seguros de salud. Cuantificar su impacto responde a una de las preguntas clave del proyecto.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Boxplot
sns.boxplot(x='smoker', y='charges', data=df, palette={'yes': 'coral', 'no': 'lightgreen'}, ax=axes[0])
axes[0].set_title('Costos según Condición de Fumador')
axes[0].set_xlabel('Fumador')
axes[0].set_ylabel('Costo (USD)')

# Violin plot
sns.violinplot(x='smoker', y='charges', data=df, palette={'yes': 'coral', 'no': 'lightgreen'}, ax=axes[1])
axes[1].set_title('Distribución de Costos por Fumador (Violin)')
axes[1].set_xlabel('Fumador')
axes[1].set_ylabel('Costo (USD)')

plt.tight_layout()
plt.show()

# Estadísticos
smoker_stats = df.groupby('smoker')['charges'].agg(['count', 'mean', 'median', 'std', 'min', 'max'])
print('Estadísticos por condición de fumador:')
print(smoker_stats)
print(f'\nRatio media fumador/no fumador: {smoker_stats.loc["yes","mean"] / smoker_stats.loc["no","mean"]:.2f}x')

**Interpretación:** La diferencia entre fumadores y no fumadores es **drástica y concluyente**. Los fumadores pagan en promedio **más de 3 veces** lo que pagan los no fumadores. Además, la dispersión de costos en fumadores es mucho mayor (desviación estándar de $11,494 vs $6,014), lo cual indica que dentro del grupo de fumadores hay también una variabilidad significativa, probablemente relacionada con otros factores como la edad o el IMC. Esta visualización proporciona evidencia contundente de que el tabaquismo es el factor individual con mayor impacto en el costo del seguro.

### 2.2 Relación edad vs costo

**Pregunta:** ¿Cómo se relaciona la edad con el costo del seguro?

**¿Por qué es relevante?** La edad es un factor de riesgo progresivo. Entender si la relación es lineal o no ayuda a justificar (o no) su uso en PCA como variable numérica.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

# Scatter plot con diferenciación por fumador
sns.scatterplot(x='age', y='charges', hue='smoker', data=df, 
                palette={'yes': 'coral', 'no': 'steelblue'}, alpha=0.6, s=60, ax=ax)

# Líneas de tendencia
for smoker, color in [('yes', 'darkred'), ('no', 'darkblue')]:
    subset = df[df['smoker'] == smoker]
    z = np.polyfit(subset['age'], subset['charges'], 1)
    p = np.poly1d(z)
    x_range = np.linspace(subset['age'].min(), subset['age'].max(), 100)
    ax.plot(x_range, p(x_range), color=color, linewidth=2, linestyle='--', 
            label=f'Tendencia {smoker} (pendiente: {z[0]:.1f})')

ax.set_title('Relación Edad vs Costo de Seguro')
ax.set_xlabel('Edad')
ax.set_ylabel('Costo (USD)')
ax.legend()
plt.tight_layout()
plt.show()

# Correlación
corr = df['age'].corr(df['charges'])
corr_smoker_yes = df[df['smoker']=='yes']['age'].corr(df[df['smoker']=='yes']['charges'])
corr_smoker_no = df[df['smoker']=='no']['age'].corr(df[df['smoker']=='no']['charges'])
print(f'Correlación edad-costos (general): {corr:.3f}')
print(f'Correlación edad-costos (fumadores): {corr_smoker_yes:.3f}')
print(f'Correlación edad-costos (no fumadores): {corr_smoker_no:.3f}')

**Interpretación:** La edad muestra una **correlación positiva moderada** con los costos (r = 0.305). Sin embargo, lo más revelador es la interacción con el tabaquismo: la pendiente de la tendencia para fumadores es mucho más pronunciada que para no fumadores. Esto significa que **el efecto de la edad sobre el costo se amplifica en fumadores**. Para los no fumadores, los costos aumentan gradualmente con la edad; para los fumadores, los costos se disparan a partir de los 40-50 años. Esto sugiere un efecto de interacción importante entre edad y tabaquismo.

---
## 3. Análisis Multivariado

### 3.1 Mapa de calor de correlaciones

**Pregunta:** ¿Qué combinación de factores explica mejor la variabilidad en los costos?

**¿Por qué es relevante?** Identificar correlaciones entre variables permite orientar el PCA y entender qué variables conviene incluir en el análisis de reducción de dimensionalidad.

In [ ]:
# Preparar variables numéricas + codificar categóricas
df_encoded = df.copy()
df_encoded['smoker_num'] = df_encoded['smoker'].map({'yes': 1, 'no': 0})
df_encoded['sex_num'] = df_encoded['sex'].map({'male': 1, 'female': 0})

# One-hot encoding para region
region_dummies = pd.get_dummies(df_encoded['region'], prefix='region')
df_encoded = pd.concat([df_encoded, region_dummies], axis=1)

# Seleccionar columnas para la matriz
cols_corr = ['age', 'bmi', 'children', 'charges', 'smoker_num', 'sex_num',
             'region_northeast', 'region_northwest', 'region_southeast', 'region_southwest']

corr_matrix = df_encoded[cols_corr].corr()

# Mapa de calor
fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, mask=mask, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8, 'label': 'Correlación'}, ax=ax)
ax.set_title('Matriz de Correlación entre Variables', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

# Correlaciones con charges
print('Correlaciones con charges (ordenadas):')
charges_corr = corr_matrix['charges'].drop('charges').sort_values(ascending=False)
for var, val in charges_corr.items():
    print(f'  {var}: {val:+.3f}')

**Interpretación:** La matriz de correlación revela hallazgos clave:

1. **`smoker` es el factor dominante**: con una correlación muy alta con `charges`, el tabaquismo explica por sí solo gran parte de la variabilidad en los costos.

2. **`age` es el segundo factor más importante**: la edad tiene una correlación positiva moderada con los costos, lo cual es esperable (a mayor edad, mayor riesgo).

3. **`bmi` tiene una correlación positiva pero baja**: el IMC por sí solo no es un predictor fuerte de costos, aunque puede interactuar con otros factores.

4. **Las variables regionales y `sex` tienen correlaciones muy débiles**: la región de residencia y el sexo no parecen ser determinantes del costo del seguro.

5. **`children` muestra correlación prácticamente nula**: el número de hijos no influye significativamente en el costo del seguro.

**Conclusión para PCA:** `age`, `bmi` y `smoker` son las variables que más contribuyen a explicar `charges`. La región y el sexo, al tener correlaciones cercanas a cero, probablemente no aporten varianza explicativa significativa en el PCA.

## 4. Resumen de hallazgos del EDA

| Pregunta | Hallazgo | Evidencia |
|---|---|---|
| ¿Cómo se distribuyen los costos? | Distribución asimétrica con cola derecha. Media > Mediana. | Histograma + boxplot de `charges` |
| ¿Influye fumar en el costo? | **Sí, drásticamente.** Fumadores pagan >3× más. | Boxplot + violin plot fumador vs costo |
| ¿Cómo se relaciona edad y costo? | Correlación positiva moderada. Efecto amplificado en fumadores. | Scatter con líneas de tendencia |
| ¿Hay diferencias regionales? | Las correlaciones son muy débiles. | Mapa de calor |
| ¿Qué factores explican la variabilidad? | `smoker` > `age` > `bmi`. Región y sexo son irrelevantes. | Matriz de correlación |

### Variables candidatas para PCA
Según la evidencia del EDA, las variables que muestran mayor relación con `charges` son:
- `smoker` (fumador)
- `age` (edad)
- `bmi` (índice de masa corporal)

Las variables `children`, `sex` y `region` muestran correlaciones casi nulas con el costo, por lo que no se espera que contribuyan significativamente en el PCA.